In [1]:
import sys

!{sys.executable} -m pip uninstall -y numpy pandas scipy
!{sys.executable} -m pip install --upgrade pip setuptools wheel
!{sys.executable} -m pip install --no-cache-dir "numpy<2" pandas scipy

Found existing installation: numpy 1.26.4
Uninstalling numpy-1.26.4:
  Successfully uninstalled numpy-1.26.4
Found existing installation: pandas 2.3.3
Uninstalling pandas-2.3.3:
  Successfully uninstalled pandas-2.3.3
Found existing installation: scipy 1.15.3
Uninstalling scipy-1.15.3:
  Successfully uninstalled scipy-1.15.3
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.6/20.6 MB 10.9 MB/s  0:00:01 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 11.2 MB/s  0:00:01eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.7/38.7 MB 10.2 MB/s  0:00:03m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [pandas]2m2/3 [pandas]


In [3]:
import sys
print(sys.executable)

/Users/tiago/Downloads/alzheimers-detection-methodologies-organized/.venv/bin/python


In [4]:
import sys
!{sys.executable} -m pip install matplotlib scikit-learn tensorflow

In [5]:
import numpy as np
import pandas as pd
import glob
import re
import shutil
import random
import tensorflow as tf
from scipy.interpolate import interp1d
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA 
from sklearn.decomposition import KernelPCA

2026-04-02 08:17:28.742183: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [6]:
print(np.__version__)
print(pd.__version__)
print(tf.__version__)

1.26.4
2.3.3
2.16.2


In [7]:
import os
print(os.getcwd())

/Users/tiago/Downloads/alzheimers-detection-methodologies-organized/notebooks/ml_based_models


In [8]:
!ls
!ls ..
!ls ../..

01_AlzheimerDetection_Data_Preprocessing.ipynb
02_AlzheimersDetection_ML_Models.ipynb
Augmented.zip
Environment            few_shot_similarity_nn ml_based_models
Alzheimer_Environment_Python_3.12.venv model
README.md                              notebooks
data                                   results


In [10]:
!unzip -n ../../data/raw/dataset_non_augmented.zip -d ../../data/raw

Archive:  ../../data/raw/dataset_non_augmented.zip


In [11]:
import glob

# Usando o caminho relativo que aponta para a pasta onde os dados foram extraídos
files = glob.glob("../../data/raw/DataBase/*/*")

In [12]:
len(files)

48

Cleaning up the files which has non-int or non-float datatype which will be replaced by the previous timestep or its next time step

In [ ]:
def clean(path):
    df = pd.read_csv(path)
    for column in df.columns:
        if df[column].dtype == 'object':
            print("Sample : ",path," feature : ",column," is uncleaned")
            df[column] = pd.to_numeric(df[column], errors='coerce')
            df[column] = df[column].fillna(method='ffill')
            df[column] = df[column].fillna(method='bfill')
    df = df.iloc[:1024,:]
    df.to_csv(path, index=False) 

In [14]:
for i in files:
    clean(i)

In [15]:
files = glob.glob("/content/DataBase/*/*")
len(files)

0

In [18]:
import shutil
import os

# Caminhos baseados na sua organização de pastas
caminho_origem = "../../data/raw/DataBase" 
caminho_destino = "../../data/raw/DataBaseOriginal"

# Se a pasta de destino já existir, apaga ela e todo o seu conteúdo para evitar o erro FileExistsError
if os.path.exists(caminho_destino):
    shutil.rmtree(caminho_destino)

# Executando a cópia "do zero"
shutil.copytree(caminho_origem, caminho_destino)

print("Cópia de segurança criada com sucesso do zero!")

Cópia de segurança criada com sucesso do zero!


In [19]:
import os

# Caminhos atuais
caminho_trabalho_atual = "../../data/raw/DataBase"
caminho_backup_atual = "../../data/raw/DataBaseOriginal"

# Novos caminhos com nomes mais explícitos
caminho_trabalho_novo = "../../data/raw/DataBase_Trabalho"
caminho_backup_novo = "../../data/raw/DataBase_Backup"

# Renomeando a pasta de trabalho (se ela existir com o nome antigo)
if os.path.exists(caminho_trabalho_atual):
    os.rename(caminho_trabalho_atual, caminho_trabalho_novo)
    print("✅ Pasta de trabalho renomeada para: DataBase_Trabalho")
else:
    print("⚠️ A pasta DataBase já foi renomeada ou não existe.")

# Renomeando a pasta de backup (se ela existir com o nome antigo)
if os.path.exists(caminho_backup_atual):
    os.rename(caminho_backup_atual, caminho_backup_novo)
    print("✅ Pasta de backup renomeada para: DataBase_Backup")
else:
    print("⚠️ A pasta DataBaseOriginal já foi renomeada ou não existe.")

✅ Pasta de trabalho renomeada para: DataBase_Trabalho
✅ Pasta de backup renomeada para: DataBase_Backup


# Data Augmentation Techniques : 
  1.) **Shifting and Noising :** Shifting in time series data refers to a
transformation of the data where the entire series is moved forward or backward in time by a certain number of periods. This can be useful in analyzing time series data, as it can help to identify patterns or trends that may not be immediately apparent when looking at the original data.Noising adds random noise to the shifted data so that the learning model will learn to ignore noisy information and filter out the relevant info from the data.

 2.) **Rolling Mean :** Rolling mean is a technique used to smooth out the data by taking the average of a fixed window of data points. This technique can be useful for reducing noise in the data and identifying trends that are relevant.


1.) Shifting 

In [20]:
def positive_shift(path,replace = False,value = 0):
    df = pd.read_csv(path)
    shift = random.randint(3,15)
    df = df.shift(shift)
    if replace:
       df = df.fillna(df.mean())
    df = df.fillna(0)      
    deviation = df.std().tolist()
    noise = np.random.normal([0]*19,deviation,19)
    df += noise/10
    return df

def negative_shift(path,replace = False,value = 0):
    df = pd.read_csv(path)
    shift = random.randint(-15,-3)
    df = df.shift(shift)
    if replace:
       df = df.fillna(df.mean())
    df = df.fillna(0)   
    deviation = df.std().tolist()
    noise = np.random.normal([0]*19,deviation,19)
    df += noise/10
    return df    

def shift(path,label,file_no,target_dir):
    pn = random.randint(0,1)
    if pn == 1:
        file_name = label+str(file_no)+".csv"
        file_name = target_dir+"/"+file_name
        df = positive_shift(path)
        print(path," -> ",file_name," Positive Shift")
        df.to_csv(file_name,index = False)
    else:
        file_name = label+str(file_no)+".csv"
        file_name = target_dir+"/"+file_name
        df = negative_shift(path)
        print(path," -> ",file_name," Negative Shift")
        df.to_csv(file_name,index = False)    


def add_shift_per_sample(path,label):
    file_no = 25
    all_files = glob.glob(path+"/*")
    for i in all_files:
        shift(i,label,file_no,path)
        file_no += 1
   

In [25]:
# Aplicando o shift para cada subconjunto de dados na pasta de trabalho
add_shift_per_sample("../../data/raw/DataBase_Trabalho/SETA", 'healthy_open')
add_shift_per_sample("../../data/raw/DataBase_Trabalho/SETB", 'healthy_closed')
add_shift_per_sample("../../data/raw/DataBase_Trabalho/SETC", 'alzeimer_open')
add_shift_per_sample("../../data/raw/DataBase_Trabalho/SETD", 'alzeimer_closed')

print("Processamento concluído em todas as pastas!")

../../data/raw/DataBase_Trabalho/SETA/healthy_open8.csv  ->  ../../data/raw/DataBase_Trabalho/SETA/healthy_open25.csv  Positive Shift
../../data/raw/DataBase_Trabalho/SETA/healthy_open9.csv  ->  ../../data/raw/DataBase_Trabalho/SETA/healthy_open26.csv  Negative Shift
../../data/raw/DataBase_Trabalho/SETA/healthy_open7.csv  ->  ../../data/raw/DataBase_Trabalho/SETA/healthy_open27.csv  Negative Shift
../../data/raw/DataBase_Trabalho/SETA/healthy_open6.csv  ->  ../../data/raw/DataBase_Trabalho/SETA/healthy_open28.csv  Positive Shift
../../data/raw/DataBase_Trabalho/SETA/healthy_open4.csv  ->  ../../data/raw/DataBase_Trabalho/SETA/healthy_open29.csv  Negative Shift
../../data/raw/DataBase_Trabalho/SETA/healthy_open5.csv  ->  ../../data/raw/DataBase_Trabalho/SETA/healthy_open30.csv  Negative Shift
../../data/raw/DataBase_Trabalho/SETA/healthy_open1.csv  ->  ../../data/raw/DataBase_Trabalho/SETA/healthy_open31.csv  Negative Shift
../../data/raw/DataBase_Trabalho/SETA/healthy_open2.csv  ->  .

2.) Rolling Mean

In [26]:
def mean_roll(path,label,file_no,target_dir):
    df = pd.read_csv(path)
    window = random.randint(3,8)
    df = df.rolling(window = 5,center = True).mean().fillna(0)
    file_name = label+str(file_no)+".csv"
    file_name = target_dir+"/"+file_name
    print(path," -> ",file_name)
    df = df.fillna(method = 'ffill')
    df = df.fillna(0)
    df.to_csv(file_name,index = False)

def add_rolling_per_sample(path,label):
    file_no = 49
    all_files = glob.glob(path+"/*")
    for i in all_files:
        mean_roll(i,label,file_no,path)
        file_no += 1

In [27]:
import shutil

# Aplicando o rolling para cada subconjunto de dados na pasta de trabalho
add_rolling_per_sample("../../data/raw/DataBase_Trabalho/SETA", 'healthy_open')
add_rolling_per_sample("../../data/raw/DataBase_Trabalho/SETB", 'healthy_closed')
add_rolling_per_sample("../../data/raw/DataBase_Trabalho/SETC", 'alzeimer_open')
add_rolling_per_sample("../../data/raw/DataBase_Trabalho/SETD", 'alzeimer_closed')

# Criando o arquivo .zip com os dados processados
shutil.make_archive('Augmented', 'zip', '../../data/raw/DataBase_Trabalho')

print("Processamento rolling concluído e arquivo Augmented.zip criado com sucesso!")

../../data/raw/DataBase_Trabalho/SETA/healthy_open35.csv  ->  ../../data/raw/DataBase_Trabalho/SETA/healthy_open49.csv
../../data/raw/DataBase_Trabalho/SETA/healthy_open34.csv  ->  ../../data/raw/DataBase_Trabalho/SETA/healthy_open50.csv
../../data/raw/DataBase_Trabalho/SETA/healthy_open36.csv  ->  ../../data/raw/DataBase_Trabalho/SETA/healthy_open51.csv


/var/folders/yp/kd81cw5n6y9cdv4bxmrqgpf80000gn/T/ipykernel_1174/2630522988.py:8: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df = df.fillna(method = 'ffill')
/var/folders/yp/kd81cw5n6y9cdv4bxmrqgpf80000gn/T/ipykernel_1174/2630522988.py:8: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df = df.fillna(method = 'ffill')
/var/folders/yp/kd81cw5n6y9cdv4bxmrqgpf80000gn/T/ipykernel_1174/2630522988.py:8: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df = df.fillna(method = 'ffill')


../../data/raw/DataBase_Trabalho/SETA/healthy_open27.csv  ->  ../../data/raw/DataBase_Trabalho/SETA/healthy_open52.csv
../../data/raw/DataBase_Trabalho/SETA/healthy_open33.csv  ->  ../../data/raw/DataBase_Trabalho/SETA/healthy_open53.csv
../../data/raw/DataBase_Trabalho/SETA/healthy_open32.csv  ->  ../../data/raw/DataBase_Trabalho/SETA/healthy_open54.csv


/var/folders/yp/kd81cw5n6y9cdv4bxmrqgpf80000gn/T/ipykernel_1174/2630522988.py:8: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df = df.fillna(method = 'ffill')
/var/folders/yp/kd81cw5n6y9cdv4bxmrqgpf80000gn/T/ipykernel_1174/2630522988.py:8: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df = df.fillna(method = 'ffill')
/var/folders/yp/kd81cw5n6y9cdv4bxmrqgpf80000gn/T/ipykernel_1174/2630522988.py:8: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df = df.fillna(method = 'ffill')


../../data/raw/DataBase_Trabalho/SETA/healthy_open26.csv  ->  ../../data/raw/DataBase_Trabalho/SETA/healthy_open55.csv
../../data/raw/DataBase_Trabalho/SETA/healthy_open30.csv  ->  ../../data/raw/DataBase_Trabalho/SETA/healthy_open56.csv
../../data/raw/DataBase_Trabalho/SETA/healthy_open25.csv  ->  ../../data/raw/DataBase_Trabalho/SETA/healthy_open57.csv


...

../../data/raw/DataBase_Trabalho/SETD/alzeimer_closed33.csv  ->  ../../data/raw/DataBase_Trabalho/SETD/alzeimer_closed65.csv
../../data/raw/DataBase_Trabalho/SETD/alzeimer_closed32.csv  ->  ../../data/raw/DataBase_Trabalho/SETD/alzeimer_closed66.csv
../../data/raw/DataBase_Trabalho/SETD/alzeimer_closed26.csv  ->  ../../data/raw/DataBase_Trabalho/SETD/alzeimer_closed67.csv
../../data/raw/DataBase_Trabalho/SETD/alzeimer_closed36.csv  ->  ../../data/raw/DataBase_Trabalho/SETD/alzeimer_closed68.csv


/var/folders/yp/kd81cw5n6y9cdv4bxmrqgpf80000gn/T/ipykernel_1174/2630522988.py:8: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df = df.fillna(method = 'ffill')
/var/folders/yp/kd81cw5n6y9cdv4bxmrqgpf80000gn/T/ipykernel_1174/2630522988.py:8: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df = df.fillna(method = 'ffill')
/var/folders/yp/kd81cw5n6y9cdv4bxmrqgpf80000gn/T/ipykernel_1174/2630522988.py:8: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df = df.fillna(method = 'ffill')
/var/folders/yp/kd81cw5n6y9cdv4bxmrqgpf80000gn/T/ipykernel_1174/2630522988.py:8: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df = df.fillna(method = '

../../data/raw/DataBase_Trabalho/SETD/alzeimer_closed35.csv  ->  ../../data/raw/DataBase_Trabalho/SETD/alzeimer_closed69.csv
../../data/raw/DataBase_Trabalho/SETD/alzeimer_closed34.csv  ->  ../../data/raw/DataBase_Trabalho/SETD/alzeimer_closed70.csv
../../data/raw/DataBase_Trabalho/SETD/alzeimer_closed8.csv  ->  ../../data/raw/DataBase_Trabalho/SETD/alzeimer_closed71.csv
../../data/raw/DataBase_Trabalho/SETD/alzeimer_closed9.csv  ->  ../../data/raw/DataBase_Trabalho/SETD/alzeimer_closed72.csv


/var/folders/yp/kd81cw5n6y9cdv4bxmrqgpf80000gn/T/ipykernel_1174/2630522988.py:8: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df = df.fillna(method = 'ffill')
/var/folders/yp/kd81cw5n6y9cdv4bxmrqgpf80000gn/T/ipykernel_1174/2630522988.py:8: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df = df.fillna(method = 'ffill')
/var/folders/yp/kd81cw5n6y9cdv4bxmrqgpf80000gn/T/ipykernel_1174/2630522988.py:8: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df = df.fillna(method = 'ffill')
/var/folders/yp/kd81cw5n6y9cdv4bxmrqgpf80000gn/T/ipykernel_1174/2630522988.py:8: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df = df.fillna(method = '

Processamento rolling concluído e arquivo Augmented.zip criado com sucesso!


**Splitting data to train and test**

In [28]:
import glob

# 1. Pegamos os arquivos originais da pasta de backup (que está intocada)
arquivos_originais_backup = glob.glob("../../data/raw/DataBase_Backup/*/*")

# Trocamos o nome no texto do caminho para que as strings batam exatamente com a pasta de trabalho
files = [caminho.replace("DataBase_Backup", "DataBase_Trabalho") for caminho in arquivos_originais_backup]

def return_splits():
    # 2. Pegamos todos os arquivos (originais + aumentados) da pasta de trabalho
    augmented_files = glob.glob("../../data/raw/DataBase_Trabalho/*/*") 
    
    # Agora a subtração de conjuntos funciona perfeitamente, isolando os novos arquivos gerados!
    train_files = list(set(augmented_files) - set(files))
    test_files = files.copy() # .copy() para evitar alterar a lista original por acidente
    
    for i in range(0, len(test_files), 12):
        train_files += test_files[i:i+4]
        
    test_files = list(set(test_files) - set(train_files))
    
    return train_files, test_files

# 3. Executamos a função
train, test = return_splits()

# Print de verificação
print(f"Quantidade de arquivos no treino: {len(train)}")
print(f"Quantidade de arquivos no teste: {len(test)}")

Quantidade de arquivos no treino: 160
Quantidade de arquivos no teste: 32


In [29]:
len(train),len(test)

(160, 32)

# Compressing Time Series Data 
PCA can be used to compress time series data of many time steps to a vector of length musch lesser by identifying the most important patterns or features in the data and representing it using a smaller set of variables or dimensions, called principal components.
Suppose we have a time series dataset with 10,000 time steps and each time step has multiple variables. We can apply PCA to this data to identify the most important patterns or features across all the variables. PCA will generate a set of principal components, with each component representing a linear combination of the original variables that captures the most variance in the data.


In [30]:
def decompose(wave_data):    
    time_series_data = wave_data.T
    kpca = KernelPCA(n_components=5, kernel='rbf', gamma=0.1)
    compressed_data = kpca.fit_transform(time_series_data).mean(axis = 1)
    return compressed_data.tolist()

In [31]:
def handle_nan(df_f):
    df = pd.read_csv(df_f)
    count = 0
    if df.isnull().sum().sum() != 0:
        print(df_f,end = " ")
        while df.isnull().sum().sum() != 0:
            count = count + 1
            df = df.fillna(method = 'ffill')
            df = df.fillna(method = 'bfill')
        print("   " , count, "null values found and taken care of","\n")    
    return df

In [32]:
def create_ds(all_files):
    dataset = []
    for file in all_files:
        df = handle_nan(file)
        scaler = StandardScaler()
        df = scaler.fit_transform(df)
        sample = decompose(df)

        if "closed" in file:
            sample.append("closed")
        elif "open" in file:
            sample.append("open")

        if "healthy" in file:
            sample.append("healthy")
        elif "alzeimer" in file:
            sample.append("alzeimer")

        dataset.append(sample)
    return pd.DataFrame(dataset)        

In [33]:
train_df = create_ds(train)

../../data/raw/DataBase_Trabalho/SETC/alzeimer_open10.csv     1 null values found and taken care of 

../../data/raw/DataBase_Trabalho/SETC/alzeimer_open8.csv     1 null values found and taken care of 



/var/folders/yp/kd81cw5n6y9cdv4bxmrqgpf80000gn/T/ipykernel_1174/1764632834.py:8: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df = df.fillna(method = 'ffill')
/var/folders/yp/kd81cw5n6y9cdv4bxmrqgpf80000gn/T/ipykernel_1174/1764632834.py:9: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df = df.fillna(method = 'bfill')
/var/folders/yp/kd81cw5n6y9cdv4bxmrqgpf80000gn/T/ipykernel_1174/1764632834.py:8: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df = df.fillna(method = 'ffill')
/var/folders/yp/kd81cw5n6y9cdv4bxmrqgpf80000gn/T/ipykernel_1174/1764632834.py:9: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df = df.fillna(method = '

In [34]:
train_df.to_csv("train.csv",index= False)

In [35]:
test_df = create_ds(test)

../../data/raw/DataBase_Trabalho/SETC/alzeimer_open6.csv     1 null values found and taken care of 

../../data/raw/DataBase_Trabalho/SETC/alzeimer_open5.csv     1 null values found and taken care of 



/var/folders/yp/kd81cw5n6y9cdv4bxmrqgpf80000gn/T/ipykernel_1174/1764632834.py:8: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df = df.fillna(method = 'ffill')
/var/folders/yp/kd81cw5n6y9cdv4bxmrqgpf80000gn/T/ipykernel_1174/1764632834.py:9: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df = df.fillna(method = 'bfill')
/var/folders/yp/kd81cw5n6y9cdv4bxmrqgpf80000gn/T/ipykernel_1174/1764632834.py:8: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df = df.fillna(method = 'ffill')
/var/folders/yp/kd81cw5n6y9cdv4bxmrqgpf80000gn/T/ipykernel_1174/1764632834.py:9: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df = df.fillna(method = '

In [36]:
test_df.to_csv("test.csv",index= False)